# Diffie-Hellman: compartilhamento de segredo

Duas partes querem combinar um segredo `s` conversando por um canal público.

1. Ambas concordam publicamente com um primo `p` e uma raiz primitiva `g` módulo `p`.
2. Um lado escolhe um segredo `a` e envia `A = g^a mod p`.
3. O outro lado escolhe um segredo `b` e envia `B = g^b mod p`.
4. Quem tem `a` calcula `s = B^a mod p`; quem tem `b` calcula `s = A^b mod p`.

Como `B^a = (g^b)^a = g^(ab) = (g^a)^b = A^b (mod p)`, os dois lados chegam ao mesmo `s`.
Quem só vê `p, g, A, B` precisaria resolver o logaritmo discreto para descobrir `a` ou `b`.

## 1. Parâmetros escolhidos

- `p = 2147483647` = 2^31 - 1 (primo de Mersenne)
- `g = 7` (raiz primitiva módulo `p`, verificada abaixo)
- `a`, `b`: chaves privadas de cada lado

In [1]:
p = 2147483647  # primo (2^31 - 1)
g = 7           # raiz primitiva módulo p

a = 123456789   # chave privada do lado A
b = 987654321   # chave privada do lado B

print(f"p = {p}")
print(f"g = {g}")
print(f"a = {a}  (privado)")
print(f"b = {b}  (privado)")

p = 2147483647
g = 7
a = 123456789  (privado)
b = 987654321  (privado)


## 2. Verificando que `p` é primo e `g` é raiz primitiva

`g` é raiz primitiva módulo `p` se sua ordem é `p - 1`, ou seja, se
`g^((p-1)/q) mod p != 1` para todo fator primo `q` de `p - 1`.

In [2]:
def eh_primo(n):
    """Teste de primalidade por divisão (suficiente para p ~ 2^31)."""
    if n < 2:
        return False
    if n % 2 == 0:
        return n == 2
    d = 3
    while d * d <= n:
        if n % d == 0:
            return False
        d += 2
    return True


def fatores_primos(n):
    """Conjunto dos fatores primos de n."""
    fatores = set()
    d = 2
    while d * d <= n:
        while n % d == 0:
            fatores.add(d)
            n //= d
        d += 1
    if n > 1:
        fatores.add(n)
    return fatores


def eh_raiz_primitiva(g, p):
    return all(pow(g, (p - 1) // q, p) != 1 for q in fatores_primos(p - 1))


assert eh_primo(p), "p não é primo"
assert eh_raiz_primitiva(g, p), "g não é raiz primitiva módulo p"

print(f"p é primo: {eh_primo(p)}")
print(f"fatores primos de p-1: {sorted(fatores_primos(p - 1))}")
print(f"g = {g} é raiz primitiva módulo p: {eh_raiz_primitiva(g, p)}")

p é primo: True
fatores primos de p-1: [2, 3, 7, 11, 31, 151, 331]
g = 7 é raiz primitiva módulo p: True


## 3. Valores públicos trocados

Cada lado calcula e envia seu valor público: `A = g^a mod p` e `B = g^b mod p`.

In [3]:
A = pow(g, a, p)  # enviado por quem tem a
B = pow(g, b, p)  # enviado por quem tem b

print(f"A = g^a mod p = {A}")
print(f"B = g^b mod p = {B}")

A = g^a mod p = 510444705
B = g^b mod p = 1018773341


## 4. Segredo compartilhado

Cada lado calcula `s` com sua chave privada e o valor público recebido.

In [4]:
s_a = pow(B, a, p)  # lado com a usa o B recebido
s_b = pow(A, b, p)  # lado com b usa o A recebido

assert s_a == s_b, "os segredos não coincidem!"
s = s_a

print(f"s (via a) = B^a mod p = {s_a}")
print(f"s (via b) = A^b mod p = {s_b}")
print(f"Segredos iguais: {s_a == s_b}")

s (via a) = B^a mod p = 366175671
s (via b) = A^b mod p = 366175671
Segredos iguais: True


## 5. Resumo dos valores

In [5]:
print("Públicos:")
print(f"  p = {p}")
print(f"  g = {g}")
print(f"  A = {A}")
print(f"  B = {B}")
print("Privados:")
print(f"  a = {a}")
print(f"  b = {b}")
print("Segredo compartilhado:")
print(f"  s = {s}")

Públicos:
  p = 2147483647
  g = 7
  A = 510444705
  B = 1018773341
Privados:
  a = 123456789
  b = 987654321
Segredo compartilhado:
  s = 366175671
